### KNN Classifier Model

In [1]:
%run data_prep.ipynb 

In [2]:
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler, LabelEncoder  
from sklearn.neighbors import KNeighborsClassifier      

le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

# scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# train a simple KNN model
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)

print("Baseline Accuracy:", accuracy_score(y_test, y_pred))


Baseline Accuracy: 0.6878246415956784


### KNN Classifier Parameter tuning

In [3]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, f1_score

# define parameter grid
param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11, 15, 20, 25],
    'weights': ['uniform', 'distance'],  # uniform = equal weights; distance = closer neighbors more weight
    'p': [1, 2]  # 1 = Manhattan distance, 2 = Euclidean distance
}

# use multiple scoring metrics
scoring = {
    'accuracy': 'accuracy',
    'f1_macro': make_scorer(f1_score, average='macro')
}

print('Running Grid Search...')

# do Grid Search optimizing for F1 score
grid = GridSearchCV(
    estimator=KNeighborsClassifier(),
    param_grid=param_grid,
    scoring=scoring,
    refit='f1_macro',  # choose thhe best model based on F1 
    cv=5,
    n_jobs=-1,
    verbose=2
)

grid_search = grid.fit(X_train, y_train)



Running Grid Search...
Fitting 5 folds for each of 32 candidates, totalling 160 fits
[CV] END ................n_neighbors=3, p=1, weights=uniform; total time=  23.5s
[CV] END ................n_neighbors=3, p=1, weights=uniform; total time=  23.6s
[CV] END ................n_neighbors=3, p=1, weights=uniform; total time=  23.6s
[CV] END ...............n_neighbors=3, p=1, weights=distance; total time=  23.6s
[CV] END ...............n_neighbors=3, p=1, weights=distance; total time=  23.7s
[CV] END ................n_neighbors=3, p=1, weights=uniform; total time=  23.7s
[CV] END ................n_neighbors=3, p=1, weights=uniform; total time=  23.7s
[CV] END ...............n_neighbors=3, p=1, weights=distance; total time=  24.1s
[CV] END ................n_neighbors=3, p=2, weights=uniform; total time=   3.6s
[CV] END ................n_neighbors=3, p=2, weights=uniform; total time=   3.7s
[CV] END ................n_neighbors=3, p=2, weights=uniform; total time=   3.7s
[CV] END ...............

In [4]:
# view our best parameters
print("Best parameters found:", grid_search.best_params_)
print("Best Cross-Validation F1-score:", grid_search.best_score_)

# evaluate on test set
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)

print("Test Accuracy:", accuracy_score(y_test, y_pred_best))
print("Test F1-score:", f1_score(y_test, y_pred_best, average='macro'))

best_model


Best parameters found: {'n_neighbors': 25, 'p': 1, 'weights': 'distance'}
Best Cross-Validation F1-score: 0.7061952563045254
Test Accuracy: 0.7167047579472262
Test F1-score: 0.7108564711970381


KNeighborsClassifier(n_neighbors=25, p=1, weights='distance')